# 015 — Conservative D cores: high precision at the cost of boundary recall

Постобработка вероятностей слабых моделей 009–013: модель/среднее/минимум/геометрическое среднее/вторая по величине вероятность; пороги, минимальная длительность D и симметричный отступ от границ каждого D-фрагмента. Отступы 0/1/2/4 секунды; это общие правила, не исправления отдельных кадров матча.

Выбор максимального recall при precision>=0.95 или 0.99 и минимум 100 предсказанных D-кадров на первых четырёх публичных минутах. Последняя минута не участвует в выборе. Фильтрация офлайн на полном контексте входных данных; метки пятой минуты не используются до фиксации настроек. Пять минут — смешанная оценка с подбором; последняя минута ранее многократно исследовалась, поэтому не независимый тест.

Run All после 014. Ноутбук использует определения и вероятности из первой части 014, не запускает его подбор. При отсутствии локальных вероятностей восстанавливает их из опубликованных моделей родителей. Исходные данные остаются локальными. Новых весов нет. Публикуется конфигурация и код. Сохраняются D/LIVE/X; это не конкурсная H/A/D заявка.


In [ ]:
from pathlib import Path
import json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'games.py').exists())
source=''.join(json.loads((ROOT/'sol/014_dead_precision_calibration/experiment.ipynb').read_text())['cells'][1]['source'])
# Load model probabilities and common functions, but do not execute the parent selection.
# Загружаем вероятности и функции, но не исполняем подбор родительского эксперимента.
exec(source.split('val_ids=np.arange(START,START+6000)')[0].replace("EXPERIMENT='014_dead_precision_calibration'","EXPERIMENT='015_dead_conservative_cores'"))
EROSIONS=[0,25,50,100]
selection_mask=public&(positions<START+6000)
records=[];best={}
def conservative_cores(mask,margin):
 result=np.zeros_like(mask)
 for a,b in spans(mask):
  if mask[a] and b-a>2*margin:result[a+margin:b-margin]=True
 return result
# All choices fixed before the last-minute assessment.
# Все настройки фиксируются до оценки последней минуты.
for name,p in tqdm(combined.items(),desc='Conservative cores / Надёжные середины'):
 p=np.where(graded,p,-1.)
 for threshold in THRESHOLDS:
  for duration in MIN_DURATIONS:
   base=filter_segment(p>=threshold,duration)
   for margin in EROSIONS:
    prediction=conservative_cores(base,margin)
    row=score(truth[selection_mask],prediction[selection_mask]);row.update(model=name,threshold=threshold,min_dead_frames=duration,margin_frames=margin)
    records.append(row)
    for target in TARGETS:
     if row['D_precision']<target or row['predicted_D_frames']<MIN_PREDICTED:continue
     key=(row['D_recall'],row['D_precision'],-margin)
     if str(target) not in best or key>best[str(target)][0]:best[str(target)]=(key,row)
selected={str(target):dict(best[str(target)][1],feasible=True) if str(target) in best else dict(model='mean',threshold=1.1,min_dead_frames=25,margin_frames=0,feasible=False,D_precision=0.,D_recall=0.,predicted_D_frames=0) for target in TARGETS}
save_json(OUT/'selected_before_test.json',selected)
outputs={}
for target in TARGETS:
 choice=selected[str(target)];p=combined[choice['model']]
 pred=np.zeros(N,bool)
 for _,a,b in BLOCKS:pred[a:b+1]=conservative_cores(filter_segment(p[a:b+1]>=choice['threshold'],choice['min_dead_frames']),choice['margin_frames'])
 tag=f'cores_p{int(target*100)}';labels=np.where(graded,np.where(pred,'D','LIVE'),'X')
 pd.DataFrame({'frame':np.arange(N),'label':labels}).to_csv(OUT/f'{tag}_predictions.csv',index=False)
 np.save(OUT/f'{tag}_dead_mask.npy',pred);np.save(OUT/f'{tag}_p_D.npy',p)
 metrics={name:score(truth[mask],pred[mask]) for name,mask in [('public_5min',public),('last_minute',public&(positions>=START+6000))]}
 # Ensure validation and inference contexts agree on the selected prefix score.
 # Проверяем совпадение контекста подбора и итогового применения на префиксе.
 if choice['feasible']:
  check=score(truth[selection_mask],pred[selection_mask])
  assert check['true_positive']==choice['true_positive'] and check['false_positive']==choice['false_positive']
 outputs[str(target)]=dict(selection=choice,metrics=metrics,prediction_file=f'{tag}_predictions.csv')
save_json(OUT/'results.json',dict(outputs=outputs,selection_window='public first four minutes',evaluation_context='full offline inputs, prefix labels only for selection; last minute diagnostic'))
pd.DataFrame(records).to_csv(OUT/'development_threshold_search.csv',index=False)
save_json(OUT/'configuration.json',dict(parents=parents,selected=selected,thresholds=THRESHOLDS,minimum_durations=MIN_DURATIONS,erosion_frames=EROSIONS,selection_range=[START,START+6000],diagnostic_range=[START+6000,END]))
progress(stage='completed',configurations=len(records))
print(json.dumps(outputs,indent=2))
